In [ ]:
!python -m pip install --user datasets

In [1]:
import pandas as pd
from datasets import load_dataset


C:\Users\mansh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# !pip install -U g4f
# !pip install aiohttp
!python -m pip install --user -U g4f
!python -m pip install --user aiohttp


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip



  Obtaining dependency information for g4f from https://files.pythonhosted.org/packages/05/f4/0c3dabd8e1b6b8362cddf191ef288be6598542f8861480919c7b78549433/g4f-0.4.3.6-py3-none-any.whl.metadata
     ---------------------------------------- 0.0/54.4 kB ? eta -:--:--
     ---------------------------------------- 54.4/54.4 kB 1.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---- ----------------------------------- 0.1/1.2 MB 2.8 MB/s eta 0:00:01
   ------------ --------------------------- 0.4/1.2 MB 4.6 MB/s eta 0:00:01
   ------------------- -------------------- 0.6/1.2 MB 4.4 MB/s eta 0:00:01
   --------------------------- ------------ 0.8/1.2 MB 4.6 MB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.2 MB 4.7 MB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.2 MB 4.7 MB/s eta 0:00:01
   ------------------------------------ --- 1.0/1.2 MB 4.7 MB/s eta 0:00:01
   ---------------------------------------- 1.2/1.2 M


[notice] A new release of pip is available: 23.2.1 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import time
from g4f.client import Client

# Initialize the client
client = Client()

# List of input messages
messages = [
    "Tell me a joke on ML engineer"
]

total_time = 0
retry_limit = 3  # Maximum retries for a failed request

# Loop through each message, send it, and measure response time
for i, msg in enumerate(messages):
    print(f"Processing message {i + 1} of {len(messages)}...")

    for attempt in range(retry_limit):
        try:
            start_time = time.time()  # Start the timer

            # API request
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": msg}],
                web_search=False
            )

            end_time = time.time()  # End the timer
            elapsed_time = end_time - start_time
            total_time += elapsed_time

            print(f"Message {i + 1}: {msg}")
            print(f"Response: {response.choices[0].message.content}")
            print(f"Time taken: {elapsed_time:.2f} seconds\n")
            break  # Exit retry loop on success

        except Exception as e:
            print(f"Attempt {attempt + 1} failed with error: {e}")
            if attempt + 1 == retry_limit:
                print(f"Skipping message {i + 1} after {retry_limit} attempts.\n")
            else:
                print("Retrying...\n")
                time.sleep(1)  # Add delay before retrying

# Summary
print(f"Total time for {len(messages)} messages: {total_time:.2f} seconds")
if messages:
    print(f"Average time per message: {total_time / len(messages):.2f} seconds")


Processing message 1 of 1...
Message 1: Tell me a joke on ML engineer
Response: Why did the machine learning engineer break up with their partner?

Because they had too many "overfitting" issues!
Time taken: 6.47 seconds

Total time for 1 messages: 6.47 seconds
Average time per message: 6.47 seconds


In [ ]:
# BLIP model
import asyncio
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process stories from a DataFrame
async def main():
    # Load the DataFrame (Modify the path to your CSV file if needed)
    df = pd.read_csv("C:\\Users\\mansh\\OneDrive\\Desktop\\NanoVLM inferencing\\BLIP-base_story_completions.csv")
  # Assuming CSV has 'Partial Story' and 'Completed Story' columns
    
    # Create an instance of the Client
    client = Client()
    executor = ThreadPoolExecutor(max_workers=20)
    
    tasks = []
    for idx, row in df.iterrows():
        s_no = idx + 1  # Generating sequential serial numbers
        partial_story = row["original_story"]
        completed_story = row["completed_story"]
        prompt = create_prompt(s_no, partial_story, completed_story)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial_story, completed_story, task))
    
    # Gather results
    results = await asyncio.gather(*(task[3] for task in tasks))
    
    # Store responses in DataFrame
    # df["Response"] = results
    # df.to_csv("story_evaluation_results.csv", index=False)
    
    print("\nPrinting the responses")
    for (s_no, partial, completed), response in zip([(task[0], task[1], task[2]) for task in tasks], results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  Serial Number: 20

**Child's Completed Story:**
The toilet has a big, round light above it.

**Assessment:**

- **Grammar:** 10/10
  - The sentence is grammatically correct.
  
- **Creativity:** 2/10
  - Very minimal creativity is shown, as the child just completed the sentence without adding any unique elements or imagination.

- **Consistency:** 10/10
  - The completed sentence is consistent with the partial story provided.

- **Meaningfulness:** 10/10
  - The completed sentence makes perfect sense and is meaningful.

- **Plot:** 2/10
  - There is no real plot development; it is just an observation.
  
**Total Score:** 34/50

**Estimated Age Group:** B: 4-5
Generated response:  Serial Number: 4

**Grading:**

1. **Grammar (8/10)**: The completion maintains proper grammar and structure, but it is identical to the beginning text, which may indicate a lack of expansion or variation in sentence structure.

2. **Creativity (5/10)**: The completion does not add any new

In [8]:
#git
import asyncio
import time
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process stories from a DataFrame
async def main():
    # Load the DataFrame (Modify the path to your CSV file if needed)
    df = pd.read_csv("C:\\Users\\mansh\\OneDrive\\Desktop\\NanoVLM inferencing\\git-base_story_completions.csv")
  # Assuming CSV has 'Partial Story' and 'Completed Story' columns
    
    # Create an instance of the Client
    client = Client()
    executor = ThreadPoolExecutor(max_workers=20)
    
    tasks = []
    for idx, row in df.iterrows():
        s_no = idx + 1  # Generating sequential serial numbers
        partial_story = row["original_story"]
        completed_story = row["completed_story"]
        prompt = create_prompt(s_no, partial_story, completed_story)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial_story, completed_story, task))
    
    # Gather results
    results = await asyncio.gather(*(task[3] for task in tasks))
    
    # Store responses in DataFrame
    # df["Response"] = results
    # df.to_csv("story_evaluation_results.csv", index=False)
    
    print("\nPrinting the responses")
    for (s_no, partial, completed), response in zip([(task[0], task[1], task[2]) for task in tasks], results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  Sure! Please provide the student's completed text for the given partial story so I can evaluate it accordingly.
Generated response:  **Serial Number: 10**

**Grading:**

1. **Grammar: 10/10**  
   The completion is grammatically correct. The sentence structure is appropriate for the beginning of the text.

2. **Creativity: 5/10**  
   The completion does not add any new elements or imaginative details to the story. It simply repeats the initial phrase without expanding on it.

3. **Consistency: 10/10**  
   The completed text is consistent with the beginning. It maintains the same subject and context.

4. **Meaningfulness: 8/10**  
   The completion is meaningful as it reflects the original idea, but it lacks additional context or development that could enhance its significance.

5. **Plot: 4/10**  
   There is no development of a plot in the completion. It does not introduce any action, conflict, or resolution.

**Total Score: 37/50**

**Estimated Age of the Stude

In [9]:
# kosmos-2
# for 5.65M model
import asyncio
import time
from concurrent.futures import ThreadPoolExecutor
from g4f.client import Client

# Synchronous function to send a request
def send_request_sync(client, message):
    """
    Sends a synchronous request to the GPT API.
    """
    while True:  # Retry until the request succeeds
        try:
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=[{"role": "user", "content": message}],
                web_search=False
            )
            generated_response = response.choices[0].message.content
            print("Generated response: ", generated_response)
            return generated_response
        except Exception as e:
            print(f"Error sending request: {e}")
            time.sleep(1)  # Wait for a short time before retrying

# Asynchronous function to execute the request
async def send_request(client, message, executor):
    """
    Asynchronous wrapper for the synchronous API request.
    """
    loop = asyncio.get_event_loop()
    return await loop.run_in_executor(executor, send_request_sync, client, message)

# Function to generate the prompt for grading
def create_prompt(s_no, partial_story, completed_story):
    """
    Generates the grading prompt based on the partial and completed stories.
    """
    return f"""The following exercise, the student is given a beginning of a text. The student needs to complete the partially completed text.
                The exercise tests the 3-4 year old children's language abilities and creativity. 
                Serial Number: {s_no}
                Partial Story: {partial_story}
                Completed Story: {completed_story}

                Now, grade the children's completion in terms of grammar, creativity, consistency, meaningfulness, plot each out of 10 with the text's beginning and
                completed text. INCLUDE THE SERIAL NUMBER OF THE STORY WHILE RESPONDING. Also tell the total of the scores. Moreover, please provide your best guess of what the age of the student might be,
                as reflected from the completion. Choose from possible age groups: A: 3 or under. B: 4-5. C: 6-7. D: 8-9. E:
                10-12. F: 13-16.
                """

# Main function to process a list of stories
async def main():
    # Define your list of tuples (serial number, partial_story, completed_story)
    stories = [
    ("1", "In a dark basement, there is a white", "<start> In a dark basement, there is a white toilet sitting on a wooden platform. The toilet is surrounded by a pile of rubble and debris.<pad>"),
    ("2", "In a shiny bathroom, the walls sparkle like", "<start> In a shiny bathroom, the walls sparkle like diamonds, and the floor is made of glass. In the middle of the room, there is a<pad>"),
    ("3", "There is a big table full of yummy", "<start> There is a big table full of yummy treats, including cakes, cookies, and cupcakes.<pad>"),
    ("4", "Pink cakes and lollipops rest on white tables", "<start> Pink cakes and lollipops rest on white tables at a children's birthday party.<pad>"),
    ("5", "The cake is so colorful with chocolate and", "<start> The cake is so colorful with chocolate and sprinkles.<pad>"),
    ("6", "In a funny bathroom, there are two shiny", "<start> In a funny bathroom, there are two shiny white toilets, one on the left and the other in the middle of the room.<pad>"),
    ("7", "In a happy green bathroom, there are funny", "<start> In a happy green bathroom, there are funny monkeys on the shower curtain.<pad>"),
    ("8", "The bathroom has a white toilet and a", "<start> The bathroom has a white toilet and a sink with a mirror above it. There is a blue towel on the floor next to the toilet.<pad>"),
    ("9", "In a shiny bathroom, there is a big", "<start> In a shiny bathroom, there is a big white toilet in the corner of the room.<pad>"),
    ("10", "There's a man on a shiny, old motorcycle", "<start> There's a man on a shiny, old motorcycle with a sidecar. He is wearing a suit and a tie.<pad>"),
    ("11", "There's a big building with a clock inside", "<start> There's a big building with a clock inside of it. The clock is on the side of the building, and there is a white tarp covering<pad>"),
    ("12", "The green bowl is on the table. It", "<start> The green bowl is on the table. It is filled with a large amount of broccoli.<pad>"),
    ("13", "There is a big, yummy cake on a", "<start> There is a big, yummy cake on a silver platter with blue and white frosting.<pad>"),
    ("14", "A big parade is happening! A police motorcycle", "<start> A big parade is happening! A police motorcycle and a police car are driving down the street. The police officer on the motorcycle is waving to the<pad>"),
    ("15", "A fluffy cat is on a table. It", "<start> A fluffy cat is on a table. It is sitting next to a bowl of fruit. There are bananas, apples, and oranges in the bowl.<pad>"),
    ("16", "The orange kitty sits on the table beside", "<start> The orange kitty sits on the table beside a bowl.<pad>"),
    ("17", "The kitty is very funny. It stands in", "<start> The kitty is very funny. It stands in front of the food bowls and tries to eat from one of them.<pad>"),
    ("18", "The cat is eating its food. It's funny", "<start> The cat is eating its food. It's funny to see a cat eating from a bowl.<pad>"),
    ("19", "A young man is sitting in a small", "<start> A young man is sitting in a small office cubicle, smiling at the camera. He is holding a computer mouse in his right hand, and<pad>"),
    ("20", "The toilet has a big, round light above", "<start> The toilet has a big, round light above it.<pad>"),
    ("21", "In a tiny bathroom, there is a white", "<start> In a tiny bathroom, there is a white toilet in the corner with a broken toilet paper roll on the floor next to it.<pad>"),
    ("22", "There are tiny green beads and nuts inside", "<start> There are tiny green beads and nuts inside a wooden box, along with a pair of scissors and a cup.<pad>"),
    ("23", "A man sits at his desk with a", "<start> A man sits at his desk with a foil-wrapped hot dog in his hand.<pad>"),
    ("24", "The bowl has yummy fruit like apples, bananas,", "<start> The bowl has yummy fruit like apples, bananas, and oranges in it. The cat is standing on the table and looking into the bowl.<pad>"),
    ("25", "In a big parking lot, two cool motorbikes", "<start> In a big parking lot, two cool motorbikes are parked next to each other.<pad>"),
]


    # Create an instance of the Client
    client = Client()

    # Create a ThreadPoolExecutor to run the blocking call asynchronously
    executor = ThreadPoolExecutor(max_workers=20)  # Adjust max_workers as needed for parallelism

    # Loop through the list of stories and send requests
    tasks = []
    for s_no, partial, completed in stories:
        prompt = create_prompt(s_no, partial, completed)
        task = send_request(client, prompt, executor)
        tasks.append((s_no, partial, completed, task))

    # Gather all results
    results = await asyncio.gather(*(task[3] for task in tasks))

    data = {
            "Partial": [story[1] for story in tasks],
            "Complete": [story[2] for story in tasks]
        }
    df = pd.DataFrame(data)

    # Print the DataFrame
    print("Generated DataFrame:")
    print(df)
    df.to_csv("kosmos_short_desc__results.csv")

    print("@" *100)
    print("\n printing the responses")

    # Print the final responses with serial numbers and stories
    for (s_no, partial, completed), response in zip(stories, results):
        print(f"Serial Number: {s_no}")
        print(f"Partial Story: {partial}")
        print(f"Completed Story: {completed}")
        print(f"Response: {response}")
        print("-" * 50)

# Run the main function
await main()


Generated response:  **Serial Number: 18**

**Grading:**

1. **Grammar: 8/10**  
   The completion is mostly grammatically correct, but the phrase "It's funny to see a cat eating from a bowl" could be simplified for clarity, especially for younger children.

2. **Creativity: 7/10**  
   The response shows some creativity by adding a detail about the cat eating from a bowl, but it could have included more imaginative elements or a humorous twist.

3. **Consistency: 9/10**  
   The completion is consistent with the beginning of the text, maintaining the focus on the cat and its eating behavior.

4. **Meaningfulness: 8/10**  
   The completion is meaningful and relates well to the initial statement, providing a clear image of the situation.

5. **Plot: 7/10**  
   While the plot is simple and straightforward, it lacks a more developed storyline or conflict that could enhance engagement.

**Total Score: 39/50**

**Estimated Age of the Student: B: 4-5**  
The completion reflects a level of 